# Phase 7 — Experiment Analysis (plan §21)

Reads the CSVs written by the experiment runners in `experiments/results/` and
reproduces the comparison charts from plan §21.3. Re-run the runners first:

```bash
python experiments/run_algorithm_compare.py   # A* vs RRT vs RRT*  -> algo_compare_*.csv
python experiments/run_lka_test.py            # LKA sweep          -> lka_summary.csv
```

The `experiments/make_charts.py` script renders the same figures headlessly to
`experiments/results/charts/`; this notebook is the interactive companion.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = Path.cwd().parent / 'experiments' / 'results' if (Path.cwd().name != 'experiments') else Path.cwd() / 'results'
if not RESULTS.exists():
    RESULTS = Path('results')  # when run from the experiments/ dir
print('reading from', RESULTS.resolve())

## 1. A\* vs RRT vs RRT\* (plan §20.1)

Compute time, path length, and success rate across three scenarios:
`road_open` (clean road graph), `road_detour` (a lane blocked by a hazard),
`obstacle_field` (irregular free-space obstacles).

In [ ]:
summary = pd.read_csv(RESULTS / 'algo_compare_summary.csv')
# one row per (scenario, planner): take the largest vehicle count
top = (summary.sort_values('num_vehicles')
               .groupby(['scenario', 'planner'], as_index=False).last())
top[['scenario', 'planner', 'success_rate', 'mean_time_ms', 'mean_path_length', 'std_path_length']]

In [ ]:
scenarios = sorted(top['scenario'].unique())
planners = ['astar', 'rrt', 'rrt_star']
colors = {'astar': '#2563eb', 'rrt': '#f59e0b', 'rrt_star': '#10b981'}
metrics = [('mean_time_ms', 'Mean compute time (ms, log)', True),
           ('mean_path_length', 'Mean path length (m)', False),
           ('success_rate', 'Success rate', False)]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
x = range(len(scenarios)); w = 0.25
for ax, (field, title, log) in zip(axes, metrics):
    for pi, pl in enumerate(planners):
        vals = [float(top[(top.scenario==s) & (top.planner==pl)][field].values[0])
                if not top[(top.scenario==s) & (top.planner==pl)].empty else 0.0
                for s in scenarios]
        ax.bar([xi + (pi-1)*w for xi in x], vals, w, label=pl, color=colors[pl])
    ax.set_title(title); ax.set_xticks(list(x)); ax.set_xticklabels(scenarios, rotation=15, ha='right')
    ax.grid(axis='y', alpha=0.3)
    if log: ax.set_yscale('log')
axes[0].legend(title='planner'); fig.tight_layout(); plt.show()

**Reading it:** on `road_open` A\* returns a road-legal path and is orders of
magnitude faster than RRT\*; RRT's shorter length is misleading because it cuts
straight across the one-way grid (ignores lane topology). On `road_detour` and
`obstacle_field` the lane-graph A\* has **no alternative edge** and fails (0%),
while RRT/RRT\* detour in free space. RRT\* trades ~100× the compute for a
shorter, smoother path than RRT.

## 2. Planning load vs vehicle count (plan §20.1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for (scn, pl), g in summary.groupby(['scenario', 'planner']):
    g = g.sort_values('num_vehicles')
    ax.plot(g['num_vehicles'], g['total_time_ms'], marker='o', color=colors.get(pl),
            linestyle={'astar':'-','rrt':'--','rrt_star':':'}.get(pl,'-'), label=f'{scn}/{pl}')
ax.set_xlabel('vehicles (queries)'); ax.set_ylabel('total planning time (ms, log)')
ax.set_yscale('log'); ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2); plt.show()

## 3. LKA lateral error vs speed (plan §20.2 / §21.3)

In [ ]:
lka = pd.read_csv(RESULTS / 'lka_summary.csv')
fig, ax = plt.subplots(figsize=(7, 4.5))
for ctrl, g in lka.groupby('controller'):
    g = g.sort_values('speed_kmh')
    ax.plot(g['speed_kmh'], g['rms_lateral_m'], marker='o', label=ctrl)
ax.set_xlabel('speed (km/h)'); ax.set_ylabel('RMS lateral error (m)')
ax.set_title('LKA lateral error vs speed'); ax.grid(alpha=0.3); ax.legend(title='controller'); plt.show()
lka.pivot_table(index='speed_kmh', columns='controller', values='rms_lateral_m')